# 13 · Attention 深推 · nan 现场 · RoPE 入门

> **学习目标**：把 attention 里 3 个最容易翻车的地方各看一遍 —— 为什么 `/√d`、5 种 nan 触发方式、绝对位置编码 vs RoPE。
>
> **预备**：07（attention 手撸）、05（sinusoidal PE）。
>
> **为什么重要**：现代 LLM 的 attention 看上去复杂，本质就是这 3 块的细节。出了 nan 不会查、不知道 RoPE 是什么 —— 都是停在 attention 入门没继续深入的标志。

In [ ]:
import torch, torch.nn.functional as F
import math
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. 为什么 `Q·K^T / √d_k` —— 方差论证（数值证明）

**已知**：`Q` 和 `K` 的每个维度独立、零均值、单位方差。

**那么** `Q·K = Σ_i Q_i K_i` 是 `d` 个独立 0 均值随机变量的和，方差 = `d`，标准差 = `√d`。

维度越大，未缩放的 `Q·K` 就越大；softmax 一过就接近 one-hot，**梯度几乎全为 0**。

**对策**：除以 `√d_k`，让方差回到 1。

In [ ]:
import statistics

def measure_var(d: int, n_samples: int = 1000):
    raw, scaled = [], []
    for _ in range(n_samples):
        q = torch.randn(d); k = torch.randn(d)
        raw.append((q @ k).item())
        scaled.append((q @ k / math.sqrt(d)).item())
    return statistics.variance(raw), statistics.variance(scaled)

print(f'{"d_k":>6} | {"无 scale 方差":>15} | {"有 scale 方差":>15}')
print('-' * 50)
for d in [4, 16, 64, 256, 1024]:
    rv, sv = measure_var(d)
    print(f'{d:>6} | {rv:>15.2f} | {sv:>15.2f}')
print('\n→ 无 scale: 方差 ≈ d，随维度线性涨。有 scale: 方差 ≈ 1，与维度无关。')

In [ ]:
# 视觉化：softmax 在无 scale / 有 scale 时的「锐度」
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, scale, title in [(axes[0], False, '无 / √d_k （越长越锐）'), (axes[1], True, '有 / √d_k （维度无关）')]:
    for d in [4, 64, 1024]:
        q = torch.randn(8, d)
        k = torch.randn(8, d)
        s = q @ k.T
        if scale:
            s = s / math.sqrt(d)
        # 取一行 softmax
        probs = s[0].softmax(-1)
        ax.plot(sorted(probs.tolist(), reverse=True), '-o', label=f'd={d}')
    ax.set_xlabel('排序后的概率位置'); ax.set_ylabel('probability')
    ax.set_title(title); ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()
print('左图：维度大时几乎只有一个 token 拿到全部概率（one-hot，梯度消失）')
print('右图：分布与维度无关，几个候选都有合理概率')

## 2. 5 种 nan 触发方式 —— 与逐一对策

**口诀**：见到 `loss = nan`，先查这 5 件事，绝大多数中招。

In [ ]:
# ============== nan #1：无 scale + 大维度 -> 中间值溢出，但 softmax 不会出 nan，只会梯度消失 ==============
torch.manual_seed(0)
B, L, D = 1, 4, 4096
Q = torch.randn(B, L, D) * 5      # 故意放大
K = torch.randn(B, L, D) * 5
scores_bad  = Q @ K.transpose(-2, -1)                # 无 scale
scores_good = Q @ K.transpose(-2, -1) / math.sqrt(D)
print(f'无 scale  scores 最大={scores_bad.abs().max():>10.1f}   softmax 后熵={(-scores_bad.softmax(-1) * scores_bad.softmax(-1).clamp_min(1e-30).log()).sum(-1).mean():.3f}')
print(f'有 scale  scores 最大={scores_good.abs().max():>10.1f}   softmax 后熵={(-scores_good.softmax(-1) * scores_good.softmax(-1).clamp_min(1e-30).log()).sum(-1).mean():.3f}')
print('→ 无 scale 的熵接近 0 = 分布退化 = 梯度消失（不出 nan，但学不动）')

In [ ]:
# ============== nan #2：mask 把整行都 -inf  =>  softmax 0/0  =>  NaN ==============
scores = torch.randn(1, 1, 4, 4)
mask_bad = torch.tensor([[True, True, True, True]])    # 整行都 mask 掉
scores_masked = scores.masked_fill(mask_bad, float('-inf'))
probs = scores_masked.softmax(-1)
print('全 -inf 行的 softmax:', probs[0, 0, 0].tolist(), '<- 全是 NaN')
print('对策：mask 时保证每行至少留 1 个位置不 mask；或者最后用 nan_to_num 兜底')

# 修法演示
probs_fixed = torch.where(probs.isnan(), torch.zeros_like(probs), probs)
print('修复后:', probs_fixed[0, 0, 0].tolist())

In [ ]:
# ============== nan #3：fp16 下 attention 中间值溢出 ==============
if device == 'cuda':
    Q = torch.randn(1, 1, 64, 128, device='cuda', dtype=torch.float16) * 10
    K = torch.randn(1, 1, 64, 128, device='cuda', dtype=torch.float16) * 10
    scores_fp16 = Q @ K.transpose(-2, -1)
    print(f'fp16 中间 scores 最大={scores_fp16.abs().max().item()}  含 inf: {torch.isinf(scores_fp16).any().item()}')
    print('对策：(a) 升 fp32 做 attention 中间计算（PyTorch 2.x autocast 默认这么做）')
    print('      (b) 用 bf16 替代 fp16（范围与 fp32 同）')
else:
    print('(skip) 需要 GPU 才能跑 fp16')

In [ ]:
# ============== nan #4：除零（normalize 等场景） ==============
x = torch.zeros(4)
y_bad  = x / x.norm()                       # 0 / 0 -> nan
y_good = x / x.norm().clamp_min(1e-8)        # 加 epsilon 防 0
print(f'x.norm() = {x.norm().item()}, y_bad = {y_bad.tolist()}, y_good = {y_good.tolist()}')

# ============== nan #5：log(0) ==============
logit_zero = torch.tensor([1.0, 0.0, 0.0]).softmax(-1)
print(f'\nlog(softmax) 含 0:', logit_zero, '->', logit_zero.log())
print('对策：always use log_softmax 而不是 softmax().log()；后者数值不稳。')
print('     log_softmax 的内部公式避开了直接 log 0：', F.log_softmax(torch.tensor([100., 0., 0.]), -1))

In [ ]:
# 「找 nan 在哪一层」工具：register_hook 监控梯度
import torch.nn as nn

class DebugModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 4)
        self.fc2 = nn.Linear(4, 4)

    def forward(self, x):
        h = self.fc1(x)
        # 故意制造 nan：取 log 一个可能为负的输出
        h = (h + 1e-10).log()
        return self.fc2(h)

m = DebugModel()
x = torch.randn(2, 4)
for name, p in m.named_parameters():
    p.register_hook(lambda g, n=name: print(f'  grad {n}  nan: {torch.isnan(g).any().item()}'))
y = m(x).sum()
y.backward()
print('\n→ 用 `register_hook` 是定位「nan 从哪一层涌出来」的最直接方法。生产里：F.detect_anomaly。')

## 3. 绝对位置编码 vs RoPE（旋转位置编码）

**绝对位置编码（GPT-2 用）**：`x_pos = tok_emb + pos_emb[pos]`，把位置作为「加性偏置」。**痛点**：超出训练时的 max_len 就 OOR，扩长度只能重训。

**RoPE（Llama / Qwen 用）**：直接在 Q、K 上做「按位置相关的旋转」。**优势**：
1. 数学上 `<RoPE(q,m), RoPE(k,n)> = f(q,k, m-n)` —— 自然编码**相对位置**
2. 没有「位置表」，能外推到比训练更长的上下文（YaRN 等技术能放大）

**实现核心**：把 d 维向量两两分组成复数，每个位置 m 在每个频率 θ 上旋转 `m·θ` 角度。

In [ ]:
def build_rope_cache(seq_len: int, d_head: int, base: float = 10000.0):
    """返回 (seq_len, d_head//2) 的 cos / sin 表。"""
    assert d_head % 2 == 0
    theta = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))   # (d/2,)
    pos = torch.arange(seq_len).float()                                       # (L,)
    freqs = torch.outer(pos, theta)                                            # (L, d/2)
    return freqs.cos(), freqs.sin()

def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    x: (..., L, d_head)
    cos, sin: (L, d_head/2)
    把 x 的 d/2 对相邻分量当做复数 (x_even + i*x_odd)，乘以 e^{i·m·θ}。
    """
    x_even, x_odd = x[..., 0::2], x[..., 1::2]              # 各 (..., L, d/2)
    # 复数乘法 (a+ib)(cos+i sin) = (a cos - b sin) + i(a sin + b cos)
    rot_even = x_even * cos - x_odd * sin
    rot_odd  = x_even * sin + x_odd * cos
    # 交错合并回 d_head 维
    out = torch.stack([rot_even, rot_odd], dim=-1).flatten(-2)
    return out

# Smoke：shape
L, d = 16, 64
q = torch.randn(2, L, d)            # (B, L, d)
cos, sin = build_rope_cache(L, d)
q_rot = apply_rope(q, cos, sin)
print('原 q  shape:', q.shape, '范数', q.norm(dim=-1)[0, 0].item())
print('rot q shape:', q_rot.shape, '范数', q_rot.norm(dim=-1)[0, 0].item(), '<- 旋转不改变范数（验证）')

In [ ]:
# 关键性质：RoPE 后 <q_m, k_n> 只依赖于 m-n（相对位置）
torch.manual_seed(7)
L, d = 32, 64
cos, sin = build_rope_cache(L, d)

# 取一对固定方向的 q, k，分别放到不同位置看内积
q_base = torch.randn(d)
k_base = torch.randn(d)

results = {}
for m in [0, 5, 10, 20, 30]:                       # q 的位置
    for n in [0, 5, 10, 20, 30]:                   # k 的位置
        q_m = apply_rope(q_base.view(1, 1, d), cos[m:m+1], sin[m:m+1]).squeeze()
        k_n = apply_rope(k_base.view(1, 1, d), cos[n:n+1], sin[n:n+1]).squeeze()
        score = (q_m @ k_n).item()
        results.setdefault(m - n, []).append(score)

# 同一个 m-n 差值，无论 m/n 的绝对值是多少，内积应该都很接近
print('| m-n |  内积值 (多个 m,n 对采样)')
print('-' * 50)
for diff in sorted(results.keys()):
    vals = results[diff]
    print(f'  {diff:+3d}  | ' + '  '.join(f'{v:+.4f}' for v in vals))
print('\n→ 同 m-n 差值下的内积在不同位置非常接近 —— RoPE 的「相对位置」属性证实。')

In [ ]:
# 对比 absolute PE（notebook 05 的 sinusoidal）：换位置不会得到「相对距离」
def sinusoidal_pe(L, d):
    pe = torch.zeros(L, d)
    pos = torch.arange(L).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, d, 2).float() * -(math.log(10000.0) / d))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe

pe = sinusoidal_pe(L, d)
q_base = torch.randn(d); k_base = torch.randn(d)

abs_results = {}
for m in [0, 5, 10, 20, 30]:
    for n in [0, 5, 10, 20, 30]:
        q_m = q_base + pe[m]
        k_n = k_base + pe[n]
        abs_results.setdefault(m - n, []).append((q_m @ k_n).item())

print('Absolute PE: 同 m-n 在不同 m,n 下的内积（应该 *不* 一致）')
for diff in sorted(abs_results.keys()):
    vals = abs_results[diff]
    spread = max(vals) - min(vals)
    print(f'  {diff:+3d}  | 范围: [{min(vals):+.2f}, {max(vals):+.2f}]   差值 {spread:.2f}')
print('\n→ Absolute PE 的内积同 m-n 下变化很大 —— 它编码的是绝对位置，不是相对距离。')

## 深入思考

1. **为什么 RoPE 不需要「位置表」，能外推？**
   - RoPE 是一个**函数**（按位置 m 旋转角度 m·θ），不是查表。理论上 m 可以任意大；实践中超过训练范围效果会下降，但比 absolute PE「直接 OOR」好很多。YaRN 等技术进一步外推。
2. **RoPE 为什么用 base = 10000？**
   - 与 sinusoidal PE 同源。Llama 3 把 base 改成 50w 来更好支持长上下文。
3. **`/√d_k` 是固定常数，能不能学？**
   - 理论上可以，但实践上「学一个标量」收益很小、增加不稳定性。固定就好。
4. **如果你看到「同样 prompt 训出来的模型时灵时不灵」，先查什么？**
   - (1) seed 没固定 (2) padding 与 attention mask 没对齐 (3) fp16 没 scaler (4) data shuffle 不同
5. **生产里有时见 `attention_dropout=0`，为什么？**
   - 现代 LLM（Llama 等）大多关掉 attention 内 dropout，只在 residual 加 dropout 或干脆不加（依赖数据多 + 大模型本身正则）。

改一改：把 RoPE 的 `base=10000` 改成 `base=500000`，重跑 `build_rope_cache(10000, 64)`，看是否仍能保持相对位置性质（应能，且对长上下文更稳定）。

## 自检 ✅

- [ ] 用一句话解释「为什么 `/√d_k`」（方差控制 → 防 softmax 退化）。
- [ ] 默背 5 种 nan 触发方式 + 各自对策。
- [ ] 解释 RoPE 与 absolute PE 的根本差别（函数 vs 表 / 相对 vs 绝对）。
- [ ] 手写 RoPE 旋转公式 `(a+ib) × (cos+i·sin)`。
- [ ] 给一个 attention 内部的张量形状 `(B, H, L, L)`，能说出每一维含义。

## 下一步

进入 Stage 4 → [`../stage4_专家/14_llama_block_from_scratch.ipynb`](../stage4_专家/14_llama_block_from_scratch.ipynb)